In [1]:
!pip install -q peft transformers datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 10.3 MB/s eta 0:00:00


# Prompting

Source For this Notebook:

1. [Pre-train, Prompt, and Predict: A Systematic Survey of Prompting Methods in Natural Language Processing](https://dl.acm.org/doi/full/10.1145/3560815)
2. [Huggingface LLM Prompting Guide](https://huggingface.co/docs/transformers/en/tasks/prompting)


This exercise touches on the concepts from the following new topics covered in the lecture this week:
1. Discrete Prompting:
  
  i. Zero Shot Prompting:
  
  ii. Few-Shot Prompting:

  iii. [Chain-of-thought Prompting](https://proceedings.neurips.cc/paper_files/paper/2022/hash/9d5609613524ecf4f15af0f7b31abca4-Abstract-Conference.html)



{"name": "twitter_complaints", "description": "", "data_columns": ["Tweet text", "ID"], "label_columns": {"Label": ["complaint", "no complaint"]}}

In [8]:
# ✅ Works with locally stored train.csv and test_unlabeled.csv
# (No need to download anything from Hugging Face)

import pandas as pd
from datasets import Dataset, DatasetDict

# 1️⃣ Load your local CSVs
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test_unlabeled.csv")

# 2️⃣ Standardize column names (keep consistent naming)
# Your CSV should have columns: "Tweet text" and "Label"
# Make sure to actually assign the renamed DataFrame back
train_df = train_df.rename(columns={"Label": "labels"})

# 3️⃣ Get class names (unique label strings)
classes = sorted(train_df["labels"].unique().tolist())

# 4️⃣ Convert to Hugging Face Dataset format (optional but consistent)
train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

ds = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

# 5️⃣ Few-shot sampling logic (balanced: 2 per class)
data_df = ds["train"].to_pandas()[["Tweet text", "labels"]]

# Split into complaint vs non-complaint groups
t_p11, test_p12 = data_df[data_df["labels"] == "complaint"][:2], data_df[data_df["labels"] == "complaint"][2:]
t_p21, test_p22 = data_df[data_df["labels"] != "complaint"][:2], data_df[data_df["labels"] != "complaint"][2:]

few_shot_examples = pd.concat([t_p11, t_p21]).sample(frac=1, random_state=42).reset_index(drop=True)
test_sample       = pd.concat([test_p12, test_p22]).sample(frac=1, random_state=42).reset_index(drop=True)

# 6️⃣ Print sanity checks
print("✅ Classes found:", classes)
print("\n🎯 Few-shot examples:")
print(few_shot_examples)
print("\n🧪 Test sample:")
print(test_sample)


✅ Classes found: ['complaint', 'no complaint']

🎯 Few-shot examples:
                                          Tweet text        labels
0  @EE On Rosneath Arial having good upload and d...     complaint
1  @KristaMariePark Thank you for your interest! ...  no complaint
2  If I can't get my 3rd pair of @beatsbydre powe...     complaint
3             @HMRCcustomers No this is my first job  no complaint

🧪 Test sample:
                                           Tweet text        labels
0   I'm earning points with #CricketRewards https:...  no complaint
1                   @NCIS_CBS https://t.co/eeVL9Eu3bE  no complaint
2   @msetchell Via the settings? That’s how I do i...  no complaint
3   @asblough Yep! It should send you a notificati...  no complaint
4   Looks tasty! Going to share with everyone I kn...  no complaint
5   The wait is finally over, just joined @SquareU...  no complaint
6   @ZARA_Care I've been waiting on a reply to my ...     complaint
7   @KenyaPower_Care  no power in so

In [9]:
train_df.value_counts('labels')

,count
labels,
no complaint,33
complaint,17


# Discrete Prompts/Hard Prompts ([Liu & Yuan et al., 2023](https://dl.acm.org/doi/pdf/10.1145/3560815))
These prompts are written in natural language. They can be further classified as:

1. Zero-shot prompts
  * No training examples
  * Model not trained to do this exact task
2. Few-shot prompts
  * A few (1-5) examples are added into the prompt
  * Model can generalize based on the examples

3. Chain-of-thought Prompting
  * Instead of asking for the answer the model is asked to explain its reasoning.
Often combined with few-shot prompts




In [10]:
# ============================================================
# ✅ CONTINUATION — Load and Configure Model for Text Generation
# ============================================================

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)

# Available small/medium instruction-tuned models
LLM_MODEL_OPTIONS = [
    "bigscience/bloomz-560m",               # ~0.5B params, works on CPU but slow
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",   # 1.1B params, fast and compact
    "ministral/Ministral-3b-instruct"       # ~3B params, needs GPU
]

## SELECT ONE MODEL ##
model_name = LLM_MODEL_OPTIONS[0]  # You can change this to [1] or [2]

# ============================================================
# ⚙️ Quantization config (only used when running on GPU)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# ============================================================
# 🧠 Load tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ============================================================
# 🧩 Load model (with or without quantization)
# ============================================================
if torch.cuda.is_available():
    # GPU available → enable 4-bit quantization for efficiency
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=bnb_config,
    )
else:
    # CPU fallback (no quantization)
    model = AutoModelForCausalLM.from_pretrained(model_name)

# ============================================================
# 🚀 Create text-generation pipeline
# ============================================================
pipeline_gen = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto" if torch.cuda.is_available() else None,
)

print(f"\n✅ Model and tokenizer loaded successfully!")
print(f"   - Model: {model_name}")
print(f"   - Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Device set to use cpu



✅ Model and tokenizer loaded successfully!
   - Model: bigscience/bloomz-560m
   - Device: CPU


In [11]:
# ==========================================================
# ✅ Example — Using the text-generation pipeline (clean output)
# ==========================================================

# Step 1️⃣: Define a real prompt instead of placeholder text
input_text = (
    "Classify this tweet as complaint or no complaint:\n"
    "I am so tired of my internet disconnecting every night."
)

# Step 2️⃣: Generate model output
# (max_new_tokens limits how many tokens the model adds)
generated_text_output = pipeline_gen(
    input_text,
    max_new_tokens=128,
    temperature=0.0,       # deterministic (no randomness)
    do_sample=False,       # disables sampling
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

# Step 3️⃣: Extract the generated text from the output
gen_output = generated_text_output[0]["generated_text"]

# Step 4️⃣: Remove the input text from the output (clean the response)
only_gen_output = gen_output.replace(input_text, "").strip()

# Step 5️⃣: Print clean result
print("🔹 Clean model output:\n", only_gen_output)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🔹 Clean model output:
 No complaint


# Exercise 1
Use zero-shot prompt to predict if the tweet text provided in `test_sample` has one of the following labels:


| ---- |  Meaning | Label|
| ---- | ---- | ------- |
|   1. | The tweet text is making a complaint | complaint |
|   2. | The tweet text is not making a complaint | no complaint |

Use the text_gen_pipeline to make the predictions.


In [12]:
# ==========================================================
# Zero-shot Prompting
# ==========================================================

# The task:
# Predict if a tweet in `test_sample` is a "complaint" or "no complaint"
# using only a zero-shot prompt (no examples provided).

# Step 1️⃣: Define the zero-shot prompt
zero_shot_prompt = lambda x: f"""Classify the following sentence into one of the two categories: "complaint" or "no complaint".
Sentence: {x}
Category:"""

# Step 2️⃣: Run model predictions for each tweet
consolidated_output_res_zero_shot = []

for i in range(len(test_sample)):
    print(f"Example {i+1}/{len(test_sample)}")

    raw_text = test_sample.iloc[i]["Tweet text"]
    gold_lab = test_sample.iloc[i]["labels"]

    # Construct the full prompt
    prompt_x = zero_shot_prompt(raw_text)

    # Generate model output (deterministic)
    output_x = pipeline_gen(
        prompt_x,
        max_new_tokens=8,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    # Clean the model’s output
    output_string = output_x[0]["generated_text"].replace(prompt_x, "").strip()

    # Save result (lowercase prediction)
    consolidated_output_res_zero_shot.append([raw_text, output_string.lower(), gold_lab])

# Step 3️⃣: Convert to DataFrame
output_df_zero_shot = pd.DataFrame(
    consolidated_output_res_zero_shot,
    columns=["text", "predicted_labels", "gold_labels"]
)

# Step 4️⃣: Preview results
print("\n✅ Zero-shot predictions:")
display(output_df_zero_shot.head())


Example 1/46
Example 2/46
Example 3/46
Example 4/46
Example 5/46
Example 6/46
Example 7/46
Example 8/46
Example 9/46
Example 10/46
Example 11/46
Example 12/46
Example 13/46
Example 14/46
Example 15/46
Example 16/46
Example 17/46
Example 18/46
Example 19/46
Example 20/46
Example 21/46
Example 22/46
Example 23/46
Example 24/46
Example 25/46
Example 26/46
Example 27/46
Example 28/46
Example 29/46
Example 30/46
Example 31/46
Example 32/46
Example 33/46
Example 34/46
Example 35/46
Example 36/46
Example 37/46
Example 38/46
Example 39/46
Example 40/46
Example 41/46
Example 42/46
Example 43/46
Example 44/46
Example 45/46
Example 46/46

✅ Zero-shot predictions:


,text,predicted_labels,gold_labels
0,I'm earning points with #CricketRewards https:...,no complaint,no complaint
1,@NCIS_CBS https://t.co/eeVL9Eu3bE,no complaint,no complaint
2,@msetchell Via the settings? That’s how I do i...,no complaint,no complaint
3,@asblough Yep! It should send you a notificati...,no complaint,no complaint
4,Looks tasty! Going to share with everyone I kn...,no complaint,no complaint


# Exercise 2 - Few-Shot Prompting
Improve the prompt by turning it into a few-shot prompt by adding the examples. Some examples are in the variable `few_shot_samples`, include them to your prompt as few-shot example.

In [14]:
# ==========================================================
# Few-Shot Prompting
# ==========================================================

# Step 1️⃣: Define few-shot prompt with explicit examples
few_shot_prompt = lambda x: f"""
Classify the following sentence into one of the two categories: "complaint" or "no complaint".

## Examples

Sentence: @EE On Rosenthal Arial having good upload and download speeds but terrible latency 200ms. Why is this.
Category: complaint

Sentence: @HMRCcustomers No this is my first job
Category: no complaint

Sentence: @KristaMariePark Thank you for your interest! If you decide to cancel, you can call Customer Care at 1-800-NYTIMES.
Category: no complaint

Sentence: If I can't get my 3rd pair of @beatsbydre powerbeats to work today I'm doneski man. This is a slap in my face. Your next @Bose @BoseService
Category: complaint

Sentence: {x}
Category:
"""

# Step 2️⃣: Generate predictions
consolidated_output_res_few_shot = []

for i in range(len(test_sample)):
    print(f"Example {i+1}/{len(test_sample)}")
    raw_text = test_sample.iloc[i]["Tweet text"]
    gold_lab = test_sample.iloc[i]["labels"]

    prompt_x = few_shot_prompt(raw_text)

    # Deterministic generation (no randomness)
    output_x = pipeline_gen(
        prompt_x,
        max_new_tokens=8,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    # Clean generated text
    output_string = output_x[0]["generated_text"].replace(prompt_x, "").strip()

    # Store results
    consolidated_output_res_few_shot.append([raw_text, output_string.lower(), gold_lab])

# Step 3️⃣: Convert to DataFrame
output_df_few_shot = pd.DataFrame(
    consolidated_output_res_few_shot,
    columns=["text", "predicted_labels", "gold_labels"]
)

# Step 4️⃣: Show sample predictions
print("\n✅ Few-shot predictions:")
display(output_df_few_shot.head())


Example 1/46
Example 2/46
Example 3/46
Example 4/46
Example 5/46
Example 6/46
Example 7/46
Example 8/46
Example 9/46
Example 10/46
Example 11/46
Example 12/46
Example 13/46
Example 14/46
Example 15/46
Example 16/46
Example 17/46
Example 18/46
Example 19/46
Example 20/46
Example 21/46
Example 22/46
Example 23/46
Example 24/46
Example 25/46
Example 26/46
Example 27/46
Example 28/46
Example 29/46
Example 30/46
Example 31/46
Example 32/46
Example 33/46
Example 34/46
Example 35/46
Example 36/46
Example 37/46
Example 38/46
Example 39/46
Example 40/46
Example 41/46
Example 42/46
Example 43/46
Example 44/46
Example 45/46
Example 46/46

✅ Few-shot predictions:


,text,predicted_labels,gold_labels
0,I'm earning points with #CricketRewards https:...,no complaint,no complaint
1,@NCIS_CBS https://t.co/eeVL9Eu3bE,no complaint,no complaint
2,@msetchell Via the settings? That’s how I do i...,no complaint,no complaint
3,@asblough Yep! It should send you a notificati...,no complaint,no complaint
4,Looks tasty! Going to share with everyone I kn...,no complaint,no complaint


# Exercise 3 - Chain-of-thought Reasoning

In this prompt instead of asking for the answer the model is asked to explain its reasoning.

\[Optional: Combine with few-shot prompting\]


In [15]:
# ==========================================================
# Chain-of-Thought Reasoning
# ==========================================================

# Prompt asks for a brief reasoning before giving the final label
chain_of_thought_prompt = lambda x: f"""
**Task**: Classify the following sentence into one of the two categories: "complaint" or "no complaint".

**Steps**:
1. Check for Negative Feelings: Look for words that show frustration, dissatisfaction, or disappointment.
2. Identify if there’s a Problem: See if the sentence mentions something wrong, a problem, or something the speaker is unhappy with.
3. Look for a Desire to Fix Something: Notice if the speaker wants change or improvement, even if it’s implied.
4. Make Your Decision: If the sentence has negative feelings, mentions a problem, and suggests a desire for change, classify it as a "complaint." If not, classify it as "no complaint."

**Sentence**: {x}
**Category**:
"""

consolidated_output_res_chain_of_thoughts = []

for i in range(len(test_sample)):
    print(f"Example {i+1}/{len(test_sample)}")
    raw_text = test_sample.iloc[i]["Tweet text"]
    gold_lab = test_sample.iloc[i]["labels"]
    prompt_x = chain_of_thought_prompt(raw_text)

    # Deterministic generation
    output_x = pipeline_gen(
        prompt_x,
        max_new_tokens=48,           # allow brief reasoning + label
        temperature=0.0,
        do_sample=False,
        eos_token_id=getattr(tokenizer, "eos_token_id", None),
        pad_token_id=getattr(tokenizer, "pad_token_id", None),
    )

    # Clean the model output and store
    output_string = output_x[0]["generated_text"].replace(prompt_x, "").strip()
    consolidated_output_res_chain_of_thoughts.append(
        [raw_text, output_string.lower(), gold_lab]
    )

output_df_chain_of_thought = pd.DataFrame(
    consolidated_output_res_chain_of_thoughts,
    columns=["text", "predected labels", "gold_labels"]  # keep column name as in your sheet
)

print("\n✅ Chain-of-thought predictions (sample):")
display(output_df_chain_of_thought.head())


Example 1/46
Example 2/46
Example 3/46
Example 4/46
Example 5/46
Example 6/46
Example 7/46
Example 8/46
Example 9/46
Example 10/46
Example 11/46
Example 12/46
Example 13/46
Example 14/46
Example 15/46
Example 16/46
Example 17/46
Example 18/46
Example 19/46
Example 20/46
Example 21/46
Example 22/46
Example 23/46
Example 24/46
Example 25/46
Example 26/46
Example 27/46
Example 28/46
Example 29/46
Example 30/46
Example 31/46
Example 32/46
Example 33/46
Example 34/46
Example 35/46
Example 36/46
Example 37/46
Example 38/46
Example 39/46
Example 40/46
Example 41/46
Example 42/46
Example 43/46
Example 44/46
Example 45/46
Example 46/46

✅ Chain-of-thought predictions (sample):


,text,predected labels,gold_labels
0,I'm earning points with #CricketRewards https:...,no complaint,no complaint
1,@NCIS_CBS https://t.co/eeVL9Eu3bE,no complaint,no complaint
2,@msetchell Via the settings? That’s how I do i...,no complaint,no complaint
3,@asblough Yep! It should send you a notificati...,no complaint,no complaint
4,Looks tasty! Going to share with everyone I kn...,no complaint,no complaint


# Exercise 4

(Why is this step required?)

Preprocess the generated predictions if required ?
At this stage you should have three set of predictions generated with the prompts:

1. Predictions with zero-shot prompting
2. Predictions with few-shot prompting
3. Predictions with chain-of-thought prompting



In [16]:
# ==========================================================
# Clean and Normalize Predictions
# ==========================================================
# Why? The model outputs can have extra spaces, punctuation, or casing differences.
# This step makes all predictions consistent before evaluation.

def clean_labels(df, col_name):
    df[col_name] = df[col_name].str.strip().str.replace('"', '').str.replace('.', '').str.lower()
    return df

# Clean predicted labels for all three prompting methods
output_df_zero_shot  = clean_labels(output_df_zero_shot,  "predicted_labels")
output_df_few_shot   = clean_labels(output_df_few_shot,   "predicted_labels")
output_df_chain_of_thought = clean_labels(output_df_chain_of_thought, "predected labels")  # keep same column name as in your file

print("✅ All predictions cleaned and standardized!")


✅ All predictions cleaned and standardized!


# Exercise 5

Evaluate the results predicted by different prompting methods by calculating the classification metrics (i.e. macro-averaged precision, recall and f1 scores)?

What are the conclusions drawn?

In [17]:
# ==========================================================
# Evaluate Prompting Results
# ==========================================================
# Goal: Compare performance of Zero-shot, Few-shot, and Chain-of-thought prompting
# using standard classification metrics (Precision, Recall, F1-score)

from sklearn.metrics import classification_report

# ----- ZERO-SHOT RESULTS -----
print("\n🔹 ZERO-SHOT RESULTS:")
print(classification_report(
    output_df_zero_shot["gold_labels"],
    output_df_zero_shot["predicted_labels"],
    zero_division=0
))

# ----- FEW-SHOT RESULTS -----
print("\n🔹 FEW-SHOT RESULTS:")
print(classification_report(
    output_df_few_shot["gold_labels"],
    output_df_few_shot["predicted_labels"],
    zero_division=0
))

# ----- CHAIN-OF-THOUGHT RESULTS -----
print("\n🔹 CHAIN-OF-THOUGHT RESULTS:")
print(classification_report(
    output_df_chain_of_thought["gold_labels"],
    output_df_chain_of_thought["predected labels"],  # keep consistent with your column name
    zero_division=0
))



🔹 ZERO-SHOT RESULTS:
              precision    recall  f1-score   support

   complaint       0.00      0.00      0.00        15
no complaint       0.67      1.00      0.81        31

    accuracy                           0.67        46
   macro avg       0.34      0.50      0.40        46
weighted avg       0.45      0.67      0.54        46


🔹 FEW-SHOT RESULTS:
              precision    recall  f1-score   support

   complaint       0.00      0.00      0.00        15
no complaint       0.67      1.00      0.81        31

    accuracy                           0.67        46
   macro avg       0.34      0.50      0.40        46
weighted avg       0.45      0.67      0.54        46


🔹 CHAIN-OF-THOUGHT RESULTS:
              precision    recall  f1-score   support

   complaint       0.00      0.00      0.00        15
no complaint       0.67      1.00      0.81        31

    accuracy                           0.67        46
   macro avg       0.34      0.50      0.40        46
we